# Data Ingestion

In [1]:
### Document DataStructure

from langchain_core.documents import Document


Create Dummy Document

In [2]:
# # Dummy Document

# doc = Document(
#     page_content="This is the content of the dummy document.",
#     metadata={
#         'souce':'dummy.txt',
#         'pages': 1,
#         'author': 'Avik',
#         'date_created': '10-31-2026'
#     }
# )

Create Directory and Text files if not exists

In [ ]:
# # Create dir if not exists

# import os
# os.makedirs("../data/text_files/", exist_ok=True)

# # create sample text files with desired content if file does not exist

# sample_text = {
#     "../data/text_files/machine_learing_intro.txt":"""
# TEXT TEXT TEXT
# """,

#     "../data/text_files/python_intro.txt":"""
# TEXT TEXT TEXT
# """
# }

# for filepath, content in sample_text.items():
#     with open(filepath, 'w', encoding='utf-8') as f:
#         f.write(content)

# print("SUCCESS")

### Import Text Loader and Read Text

In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding='utf-8')

# Load Content
document = loader.load()

print(document)


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='PYTHON PROGRAMMING INTRODUCTION\n\n1. What Is Python?\n\nPython is a high-level, general-purpose programming language known for\nreadable syntax and a large ecosystem of libraries. It is used in\nautomation, web development, data analysis, machine learning, artificial\nintelligence, scripting, and application development.\n\nPython is dynamically typed: a variable does not need an explicit type\ndeclaration. The interpreter determines the type of the value at\nruntime.\n\nExample:\n\n    name = "Mira"\n    age = 27\n    height = 1.68\n    is_active = True\n\nThe equals sign assigns a value to a variable. It does not mean\nmathematical equality in an assignment statement.\n\n2. Basic Data Types\n\nCommon built-in types include:\n\n-   int: whole numbers, such as 10 or -4.\n-   float: decimal numbers, such as 3.14 or 0.5.\n-   str: text, such as “Python”.\n-   bool: Boolean values, True and False.\n-   No

### Directory Loader

In [11]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)

text_docs = dir_loader.load()
print(text_docs)


100%|██████████| 2/2 [00:00<00:00, 943.92it/s]

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='PYTHON PROGRAMMING INTRODUCTION\n\n1. What Is Python?\n\nPython is a high-level, general-purpose programming language known for\nreadable syntax and a large ecosystem of libraries. It is used in\nautomation, web development, data analysis, machine learning, artificial\nintelligence, scripting, and application development.\n\nPython is dynamically typed: a variable does not need an explicit type\ndeclaration. The interpreter determines the type of the value at\nruntime.\n\nExample:\n\n    name = "Mira"\n    age = 27\n    height = 1.68\n    is_active = True\n\nThe equals sign assigns a value to a variable. It does not mean\nmathematical equality in an assignment statement.\n\n2. Basic Data Types\n\nCommon built-in types include:\n\n-   int: whole numbers, such as 10 or -4.\n-   float: decimal numbers, such as 3.14 or 0.5.\n-   str: text, such as “Python”.\n-   bool: Boolean values, True and False.\n-   No

In [13]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader, PyPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf_files",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    # loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)

pdf_docs = dir_loader.load()
print(pdf_docs)

100%|██████████| 3/3 [00:00<00:00, 100.04it/s]

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '../data/pdf_files/deep_learning_intro.pdf', 'file_path': '../data/pdf_files/deep_learning_intro.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='RAG Learning Notes\nPage 1\nDEEP LEARNING\nNeural networks, optimisation, architectures, and evaluation\n1. What is deep learning?\nDeep learning is a subfield of machine learning that uses neural networks with multiple layers to learn representations from\ndata. Rather than relying entirely on manually designed features, a network can learn useful intermediate features during\ntraining.\nApplications include image recognition, speech recognition, language modelling, recommendation, forecasting, and\ngenerative AI. Deep learning can require substantial data and compute, although smaller networks can work well for focused

# Enbedding and VectorStoreDB

In [ ]:
import numpy as np 
from sentence_transformers import SentenceTransformer 
import chromadb 
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/aviksarkar/Desktop/Python/rag_application/rag_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 18:54:26.365000 13390 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Embedding Manager

In [17]:
class EmbeddingManager:
    """Handles Document Embedding Generation Using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the Embedding Manager
        Args:
            model_name: Hugging Face model name for sentence embedding.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading the model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Successfully Loaded. Embedding Dimension : {self.model.get_sentence_embedding_dimension()}")
        except ValueError as e:
            print(f"Error Loading model {self.model_name} : {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """ Generate embeddings for a list of texts
            Args:
                texts: List of text strings to embed

            Returns:
                numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    # def get_embedding_dimension(self) -> int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_sentence_embedding_dimension()



## Initialize the Embedding Manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading the model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8969.07it/s]


Model Successfully Loaded. Embedding Dimension : 384


/var/folders/yz/83q64df112j55g1h0p6vkxph0000gn/T/ipykernel_13390/762315567.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Successfully Loaded. Embedding Dimension : {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a chromDB Vector Store"""

    def _init_(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the Vector Store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector stroe
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initilize_store()





